<a href="https://colab.research.google.com/github/Dennischegemwangi/code-base/blob/main/multilingual_named_entity_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#NER
# identifying dataset configuration (domains)
from datasets import get_dataset_config_names
xtreme_subset = get_dataset_config_names("google/xtreme")
print (f"{len(xtreme_subset)} configuration found in xtreme")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

183 configuration found in xtreme


In [2]:
#identifying PAN-X
panx_sub = [s for s in xtreme_subset if s.startswith ("PAN")]

panx_sub[:3]

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg']

In [7]:
from collections import defaultdict
from datasets import DatasetDict,load_dataset


langs = ["fr","it","de","en"]
fracs = [0.229,0.084,0.629,0.059]
#return dict if key doesn't mot exit
panx_ch = defaultdict (DatasetDict)

for lang ,frac in zip(langs,fracs):
    ds = load_dataset("google/xtreme" ,name = f"PAN-X.{lang}")


    # shuffle and down sample each split according to spoken proportion

    for split in ds:
        panx_ch[lang][split] = (ds[split]. shuffle (seed = 0).select(range(int(frac * ds[split].num_rows))))

PAN-X.fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  837kB            

PAN-X.fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PAN-X.fr/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  419kB            

PAN-X.fr/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

PAN-X.fr/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  423kB            

PAN-X.fr/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (26 outputs left out)

PAN-X.en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
import pandas as pd
pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs },index =["num_training_examples"])

,fr,it,de,en
num_training_examples,4580,1680,12580,1180


In [15]:
#performing zero-shot cross lingual transfer

element = panx_ch["de"]["train"][0]
#for key,value in element.items():
    #print(f"{key}:{value}")

#features of the dataset
for key, value in panx_ch["de"]["train"].features.items():
    print(f"{key}:{value}")

tokens:List(Value('string'))
ner_tags:List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs:List(Value('string'))


In [19]:
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print (tags)

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [21]:
#creating new column within the dataset for NER_TAGS that are easy to familiarize

def create_tag_names (batch):
    return {"ner_tags_str" : [tags.int2str(idx) for idx in batch["ner_tags"]]}

panx_de = panx_ch["de"].map(create_tag_names)



de_example = panx_de["train"][0]

pd.DataFrame([de_example["tokens"],de_example["ner_tags_str"]],['Tokens','Tags'])

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [25]:
#checking for imbalance within the tags

from collections import Counter


split2freq = defaultdict (Counter)

for split, dataset in panx_de.items():
    for row in dataset["ner_tags_str"]:
        for tag in row:
            if tag.startswith("B"):

               tag_type = tag.split("-")[1]
               split2freq[split][tag_type] +=1

pd.DataFrame.from_dict(split2freq,orient = "index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071
